In [12]:
# ===============================
# 1. Imports
# ===============================
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from scipy.signal import butter, filtfilt
import json
import os

os.chdir('/Users/aryantripathi/P300-EEG-Authentication')

# ===============================
# 2. Load A01.mat correctly
# ===============================
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

# Extract fields
X = s1['X']  # Shape: (347704, 8)
y_stim = s1['y_stim']  # Shape: (347704, 1) - stimulus labels

print("X shape:", X.shape)
print("y_stim shape:", y_stim.shape)

# ===============================
# 3. Convert to (channels, samples)
# ===============================
eeg = X.T  # Transpose to (8, 347704)
target = y_stim.flatten()  # Flatten to (347704,)

print("EEG shape (channels, samples):", eeg.shape)
print("Target shape:", target.shape)
print("Unique target values:", np.unique(target))

# ===============================
# 4. Bandpass filter
# ===============================
fs = 240

def bandpass_filter(signal, lowcut=0.1, highcut=30, fs=240, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal, padlen=0)

filtered_eeg = np.zeros_like(eeg, dtype=float)
for ch in range(eeg.shape[0]):
    filtered_eeg[ch] = bandpass_filter(eeg[ch].astype(float))

print("Filtered EEG shape:", filtered_eeg.shape)

# ===============================
# 5. Plot 1 — RAW EEG
# ===============================
eeg_channel_index = 2
plt.figure(figsize=(10,3))
plt.plot(eeg[eeg_channel_index, :2000].astype(float))
plt.title("Raw EEG Signal (Subject A01, Channel 2)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.grid(True)
plt.savefig("plots/raw_eeg_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 6. Plot 2 — RAW vs FILTERED EEG
# ===============================
plt.figure(figsize=(10,6))

plt.subplot(2,1,1)
plt.plot(eeg[eeg_channel_index, :2000].astype(float))
plt.title("Raw EEG")

plt.subplot(2,1,2)
plt.plot(filtered_eeg[eeg_channel_index, :2000])
plt.title("Filtered EEG (0.1–30 Hz)")

plt.tight_layout()
plt.savefig("plots/raw_vs_filtered_eeg_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 7. Epoch extraction (-200 ms to +800 ms)
# ===============================
baseline_samples = int(0.2 * fs)
post_samples = int(0.8 * fs)

# Find target stimulus indices (where target == 1)
target_indices = np.where(target == 1)[0]
print(f"Found {len(target_indices)} target events")

epochs = []
for idx in target_indices:
    start = idx - baseline_samples
    end = idx + post_samples
    if start >= 0 and end < filtered_eeg.shape[1]:
        epoch = filtered_eeg[:, start:end]
        epochs.append(epoch)

epochs = np.array(epochs)
print("Epochs shape (trials, channels, samples):", epochs.shape)

# ===============================
# 8. Baseline correction + averaging
# ===============================
baseline_corrected_epochs = np.zeros_like(epochs, dtype=float)

for trial_idx in range(epochs.shape[0]):
    for ch in range(epochs.shape[1]):
        epoch = epochs[trial_idx, ch, :]
        baseline = epoch[:baseline_samples].mean()
        baseline_corrected_epochs[trial_idx, ch, :] = epoch - baseline

# Average across trials
p300_template = baseline_corrected_epochs.mean(axis=0)
time = np.linspace(-0.2, 0.8, p300_template.shape[1])

print("P300 template shape (channels, samples):", p300_template.shape)

# ===============================
# 9. Plot 3 — FINAL AVERAGED P300
# ===============================
plt.figure(figsize=(10,4))
plt.plot(time, p300_template[eeg_channel_index, :])
plt.axvline(0, color='r', linestyle='--', label='Stimulus')
plt.title("Averaged P300 Waveform (Subject A01, Channel 2)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.savefig("plots/averaged_p300_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 10. SAVE PREPROCESSED DATA
# ===============================
os.makedirs('data/preprocessed', exist_ok=True)

# Save epochs
np.save('data/preprocessed/A01_epochs.npy', baseline_corrected_epochs)

# Save metadata
metadata = {
    'subject': 'A01',
    'n_epochs': len(baseline_corrected_epochs),
    'n_channels': baseline_corrected_epochs.shape[1],
    'n_samples': baseline_corrected_epochs.shape[2],
    'sampling_rate': fs,
    'channels': ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8'],
    'baseline_duration_ms': 200,
    'post_stimulus_duration_ms': 800
}

with open('data/preprocessed/A01_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Saved A01_epochs.npy")
print("✓ Saved A01_metadata.json")
print(f"✓ Total epochs saved: {baseline_corrected_epochs.shape[0]}")

X shape: (347704, 8)
y shape: ()
EEG shape (channels, samples): (8, 347704)
Target shape: (1,)
Filtered EEG shape: (8, 347704)
Epochs shape (trials, channels, samples): (0,)


/var/folders/2t/w_21ll7d26v67z1knktlhldr0000gn/T/ipykernel_82504/3281642363.py:115: RuntimeWarning: Mean of empty slice.
  p300_template = baseline_corrected_epochs.mean(axis=0)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


IndexError: tuple index out of range

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import json
import os

os.chdir('/Users/aryantripathi/P300-EEG-Authentication')

# ===============================
# Process ALL subjects A01-A08
# ===============================

subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08']
fs = 240

def bandpass_filter(signal, lowcut=0.1, highcut=30, fs=240, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal, padlen=0)

for subject in subjects:
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    
    try:
        # Load data
        data = sio.loadmat(f'data/raw/{subject}.mat')
        s1 = data['data'][0][0]
        
        X = s1['X']
        y_stim = s1['y_stim']
        
        print(f"X shape: {X.shape}")
        
        # Convert to (channels, samples)
        eeg = X.T
        target = y_stim.flatten()
        
        # Bandpass filter
        filtered_eeg = np.zeros_like(eeg, dtype=float)
        for ch in range(eeg.shape[0]):
            filtered_eeg[ch] = bandpass_filter(eeg[ch].astype(float))
        
        # Epoch extraction
        baseline_samples = int(0.2 * fs)
        post_samples = int(0.8 * fs)
        
        target_indices = np.where(target == 1)[0]
        
        epochs = []
        for idx in target_indices:
            start = idx - baseline_samples
            end = idx + post_samples
            if start >= 0 and end < filtered_eeg.shape[1]:
                epoch = filtered_eeg[:, start:end]
                epochs.append(epoch)
        
        epochs = np.array(epochs)
        
        # Baseline correction
        baseline_corrected_epochs = np.zeros_like(epochs, dtype=float)
        
        for trial_idx in range(epochs.shape[0]):
            for ch in range(epochs.shape[1]):
                epoch = epochs[trial_idx, ch, :]
                baseline = epoch[:baseline_samples].mean()
                baseline_corrected_epochs[trial_idx, ch, :] = epoch - baseline
        
        # Save
        os.makedirs('data/preprocessed', exist_ok=True)
        np.save(f'data/preprocessed/{subject}_epochs.npy', baseline_corrected_epochs)
        
        # Save metadata
        metadata = {
            'subject': subject,
            'n_epochs': len(baseline_corrected_epochs),
            'n_channels': baseline_corrected_epochs.shape[1],
            'n_samples': baseline_corrected_epochs.shape[2],
            'sampling_rate': fs,
            'channels': ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8'],
            'baseline_duration_ms': 200,
            'post_stimulus_duration_ms': 800
        }
        
        with open(f'data/preprocessed/{subject}_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ {subject}: {baseline_corrected_epochs.shape[0]} epochs saved")
        
    except Exception as e:
        print(f"✗ {subject}: Error - {str(e)}")

print(f"\n{'='*60}")
print("All subjects processed!")
print(f"{'='*60}")

In [13]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

y_field = s1['y']
print("y_field shape:", y_field.shape)
print("y_field type:", type(y_field))

y_stim = s1['y_stim']
print("y_stim shape:", y_stim.shape)
print("y_stim[0][0] shape:", y_stim[0][0].shape if y_stim.shape == (1,1) else "not nested")
print("y_stim content sample:", y_stim[0][0][:20] if hasattr(y_stim[0][0], '__getitem__') else y_stim[0][0])

y_field shape: (347704, 1)
y_field type: <class 'numpy.ndarray'>
y_stim shape: (347704, 1)
y_stim[0][0] shape: not nested


IndexError: invalid index to scalar variable.

In [11]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

# Check what's in X
X_field = s1['X']
print("X_field shape:", X_field.shape)
print("X_field type:", type(X_field))
print("X_field:", X_field)

if X_field.shape == (1, 1):
    X_data = X_field[0][0]
    print("X_data shape:", X_data.shape)
    print("X_data type:", type(X_data))

X_field shape: (347704, 8)
X_field type: <class 'numpy.ndarray'>
X_field: [[ -1.21978372  -0.32769319  -1.21995831 ...  -2.65680554   7.37202505
   -0.27890203]
 [ -1.42616224  -4.18996393  -5.65458822 ...  -9.45534598  -3.67053188
   -5.48506363]
 [ -1.31524453  -6.87075831  -8.55992067 ... -14.26798757 -11.63200572
   -8.92900433]
 ...
 [ -3.96511643   5.70748743   1.11047492 ...  -5.27998942   9.13724881
   -0.85580654]
 [ -0.5562302    6.01519786   3.32213104 ...  -0.67030739   8.2749988
    2.10576813]
 [  4.22622461   6.77563943   6.12924025 ...   5.20342677   6.29436858
    6.15256883]]


In [9]:
import scipy.io as sio
import numpy as np

data = sio.loadmat('data/raw/A01.mat')
s1 = data['data']

print("s1 shape:", s1.shape)
print("s1 type:", type(s1))
print("s1[0][0] fields:", s1[0][0].dtype.names if hasattr(s1[0][0], 'dtype') else 'not a struct')

# Try to access X directly
try:
    X = s1['X']
    print("X shape:", X.shape)
except:
    try:
        X = s1[0][0]['X']
        print("X[0][0] shape:", X.shape)
    except:
        print("Cannot access X")

s1 shape: (1, 1)
s1 type: <class 'numpy.ndarray'>
s1[0][0] fields: ('channels', 'X', 'y', 'y_stim', 'trial', 'classes', 'classes_stim', 'gender', 'age', 'ALSfrs', 'onsetALS')
X shape: (1, 1)


In [6]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'data'])


In [7]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data']
print(s1.dtype)
print(s1.shape)

[('channels', 'O'), ('X', 'O'), ('y', 'O'), ('y_stim', 'O'), ('trial', 'O'), ('classes', 'O'), ('classes_stim', 'O'), ('gender', 'O'), ('age', 'O'), ('ALSfrs', 'O'), ('onsetALS', 'O')]
(1, 1)
